* [#22](https://github.com/salgo60/ifkdb/issues/22)
* Matcha SPARQL sökning fotbollsspelare med bilder
* Denna Notebook [22_soccerplayer.ipynb](https://github.com/salgo60/spa2Commons/blob/main/Notebook/22_soccerplayer.ipynb)

In [1]:
from datetime import datetime
start_time  = datetime.now()
print("Last run: ", start_time)

Last run:  2026-02-27 12:04:14.430978


In [2]:
import urllib3, json
import pandas as pd 
http = urllib3.PoolManager() 

url= "https://portrattarkiv.se/endpoints/search.php"  

In [3]:
import sys
from SPARQLWrapper import SPARQLWrapper, JSON

endpoint_url = "https://query.wikidata.org/sparql"

# svenska fotbollsspelare födda innan 1920 utan bild - SPA innehåller mest gamla personer
querySoccer = """SELECT 
  (CONCAT(?itemLabel, " ", STR(YEAR(?birthDate))) AS ?search) 
  ?item 
  ?itemLabel 
  (STR(YEAR(?birthDate)) AS ?year)
WHERE {

  ?item wdt:P569 ?birthDate .
  ?item wdt:P27 wd:Q34 .
  ?item wdt:P106 wd:Q937857 .

  MINUS { ?item wdt:P19 ?img }   # saknar födelseort (som i originalet)

  FILTER(YEAR(?birthDate) < 1940)

  SERVICE wikibase:label { 
    bd:serviceParam wikibase:language "sv,en". 
  }
}
ORDER BY ?itemLabel"""


 
def get_sparql_dataframe(endpoint_url, query):
    """
    Helper function to convert SPARQL results into a Pandas data frame.
    """
    user_agent = "salgo60@msn.com/%s.%s" % (sys.version_info[0], sys.version_info[1])
 
    sparql = SPARQLWrapper(endpoint_url, agent=user_agent)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    result = sparql.query()

    processed_results = json.load(result.response)
    cols = processed_results['head']['vars']

    out = []
    for row in processed_results['results']['bindings']:
        item = []
        for c in cols:
            item.append(row.get(c, {}).get('value'))
        out.append(item)

    return pd.DataFrame(out, columns=cols)

Soccerdb = get_sparql_dataframe(endpoint_url, querySoccer)

Soccerdb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 227 entries, 0 to 226
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   search     227 non-null    object
 1   item       227 non-null    object
 2   itemLabel  227 non-null    object
 3   year       227 non-null    object
dtypes: object(4)
memory usage: 7.2+ KB


In [4]:
Soccerdb

,search,item,itemLabel,year
0,Birger Persson 1892,http://www.wikidata.org/entity/Q115805723,Birger Persson,1892
1,Einar Hemming 1893,http://www.wikidata.org/entity/Q47516929,Einar Hemming,1893
2,Erik Alstam 1889,http://www.wikidata.org/entity/Q135257517,Erik Alstam,1889
3,Knut Johansson 1902,http://www.wikidata.org/entity/Q109316247,Knut Johansson,1902
4,Sune Zetterberg 1907,http://www.wikidata.org/entity/Q109301373,Sune Zetterberg,1907
...,...,...,...,...
222,Yngve Jansson 1929,http://www.wikidata.org/entity/Q126453910,Yngve Jansson,1929
223,Åke Kling 1924,http://www.wikidata.org/entity/Q126084216,Åke Kling,1924
224,Åke Olsson 1927,http://www.wikidata.org/entity/Q19399965,Åke Olsson,1927
225,Åke Olsson 1926,http://www.wikidata.org/entity/Q48816178,Åke Olsson,1926


In [5]:
import urllib3
http = urllib3.PoolManager()
SPAdetail = "https://portrattarkiv.se/details/"
for index, row  in Soccerdb.iterrows():
    print("\n",row["search"],row["item"])
    encoded_body = json.dumps({
        "limit": "5",
        "from": "0",
        "birthyear":row["year"],
        "all":row["search"]
    })
    
    r = http.request('POST', url,
                 headers={'Content-Type': 'application/json'},
                 body=encoded_body)

    if r.status != 200:
        print(r.status)
        continue
    
    data = json.loads(r.data.decode('utf-8'),) 
    urls = []


    for h in data["hits"]["hits"]:
        id = h["_id"]
        #print(h)
        source = h["_source"]
        try:
 #           urlPicture = urlbasePic + id
 #           urls.append(urlPicture)
            score = h["_score"]
            FirstName = source["FirstName"]
            LastName = source["LastName"]
            BirthYear = source["BirthYear"]
            print("\t\t",score,FirstName, " ", LastName, " - ", BirthYear,SPAdetail+id, "\t", )
        except:
            print("Error")


 Birger Persson 1892 http://www.wikidata.org/entity/Q115805723
		 17.544235 Filip   Persson  -  1892 https://portrattarkiv.se/details/sj9PGLAlnmUAAAAAACcRmg 	
		 16.660065 Birger   Andersson  -  1892 https://portrattarkiv.se/details/sj9PGLAlnmUAAAAAAA5sCQ 	
		 16.581795 Birger   Renhorn  -  1892 https://portrattarkiv.se/details/CaU9ScYzgeAAAAAAAAAE5w 	
		 16.569618 Birger   Mattsson  -  1892 https://portrattarkiv.se/details/sj9PGLAlnmUAAAAAAB6tiw 	
		 16.526127 Birger   Bellander  -  1892 https://portrattarkiv.se/details/TNQaWo324IAAAAAAAACipg 	

 Einar Hemming 1893 http://www.wikidata.org/entity/Q47516929
		 19.884172 Hemming   Olsson  -  1893 https://portrattarkiv.se/details/vfWYdYT6EoAAAAAAAABAWA 	
		 16.12376 Einar   Peterson  -  1893 https://portrattarkiv.se/details/sj9PGLAlnmUAAAAAAA60Cw 	
		 15.928924 Einar   Hellsén  -  1893 https://portrattarkiv.se/details/j6S8rJvx9lAAAAAAAACjWw 	
		 15.873774 Einar   Moström  -  1893 https://portrattarkiv.se/details/sj9PGLAlnmUAAAAAAB6tjA 	


In [6]:
def score_class(score):
    if score < 10:
        return "score-low"
    elif score < 20:
        return "score-mid"
    else:
        return "score-high"



In [7]:
import urllib3
import json
from datetime import datetime
import os

http = urllib3.PoolManager()

SPAdetail = "https://portrattarkiv.se/details/"
github_issue = "https://github.com/salgo60/ifkdb/issues/22"

timestamp = datetime.now().strftime("%Y%m%d %H:%M")
today = datetime.now().strftime("%Y-%m-%d")
output_file = f"rapport_{today}.html"

html_rows = []

for index, row in Soccerdb.iterrows():

    encoded_body = json.dumps({
        "limit": "5",
        "from": "0",
        "birthyear": row["year"],
        "all": row["search"]
    })

    r = http.request(
        'POST',
        url,
        headers={'Content-Type': 'application/json'},
        body=encoded_body
    )

    if r.status != 200:
        print("Fel status:", r.status)
        continue

    data = json.loads(r.data.decode('utf-8'))

    for h in data["hits"]["hits"]:
        try:
            source = h["_source"]
            id = h["_id"]

            wikidata_url = row["item"]
            qid = wikidata_url.rstrip("/").split("/")[-1]
            
            html_link = f'<a target="_blank" href="https://www.wikidata.org/entity/{qid}">{qid}</a>'
            qid = wikidata_url.rstrip("/").split("/")[-1]
            correct_url = f"https://www.wikidata.org/entity/{qid}"
            score = h["_score"]
            FirstName = source.get("FirstName", "")
            LastName = source.get("LastName", "")
            BirthYear = source.get("BirthYear", "")

            detail_link = SPAdetail + id

            html_rows.append(f"""
            <tr>
                <td>{row["search"]}</td>
                <td>{html_link}</td>
                <td class="{score_class(score)}">{score:.2f}</td>
                <td>{FirstName}</td>
                <td>{LastName}</td>
                <td>{BirthYear}</td>
                <td>
                  <a href="{detail_link}" target="_blank">
                    Visa porträtt
                  </a>
                </td>
            </tr>
            """)

        except Exception as e:
            print("Fel vid rad:", e)
            continue


html_content = f"""
<!DOCTYPE html>
<html lang="sv">
<head>
<meta charset="UTF-8">
<title>Porträttarkiv Rapport</title>
<style>
    body {{ font-family: Arial, sans-serif; }}
    table {{ border-collapse: collapse; width: 100%; }}
    th, td {{ border: 1px solid #ccc; padding: 8px; text-align: left; }}
    th {{ background-color: #f2f2f2; }}
    tr:nth-child(even) {{ background-color: #fafafa; }}
    a {{ color: #0066cc; text-decoration: none; }}
    a:hover {{ text-decoration: underline; }}
    .score-low {{
      background-color: #f8d7da;   /* ljusröd */
    }}
    
    .score-mid {{
      background-color: #fff3cd;   /* gul */
    }}
    
    .score-high {{
      background-color: #d4edda;   /* grön */
    }}

</style>
</head>
<body>
<h1>Porträttarkiv – Matchningsrapport svenska äldre fotbollsspelare</h1>
<p><strong>Skapad:</strong> {timestamp}</p>
<p>
Relaterat ärende:
<a href="{github_issue}" target="_blank">
GitHub Issue #22
</a>
</p>
<table>
<thead>
<tr>
    <th>Sökning</th>
    <th>Wikidata</th>
    <th>Score</th>
    <th>Förnamn</th>
    <th>Efternamn</th>
    <th>Födelseår</th>
    <th>Länk</th>
</tr>
</thead>
<tbody>
{''.join(html_rows)}
</tbody>
</table>
</body>
</html>
"""

with open(output_file, "w", encoding="utf-8") as f:
    f.write(html_content)

print("Rapport skapad:", os.path.abspath(output_file))

Rapport skapad: /Users/salgo/Documents/GitHub/spa2Commons/Notebook/rapport_2026-02-27.html


In [8]:
print(source.keys())

dict_keys(['FirstName', 'LastName', 'BirthDate', 'BirthYear', 'Path', 'Filename', 'Size', 'ChangedDate', 'FileHash', 'Facts'])


In [9]:
print(source["Path"])
print(source["Filename"])

/ arkiv/--Porträttarkiv/P/pe/pettersson/Pettersson P/Pettersson Per 19240830-.jpg
Pettersson Per 19240830-.jpg
